# Why My R² Wouldn't Sit Still

Companion notebook to the post *My R² Wouldn't Sit Still*.

**The question:** how much of the variation I saw could be explained by the
train/test split alone?

**The method:** fit the same model 200 times, changing nothing except the
random split, and look at the distribution of test-set R² that comes out.

**Data:** the medical cost personal dataset — 1,338 rows, one per policyholder
(age, sex, BMI, children, smoker, region, annual charges). Widely mirrored;
I used the Kaggle copy. Save it as `insurance.csv` next to this notebook.

Run this top to bottom on a fresh kernel. That instruction is not decorative —
it's the entire point of the post.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

import sklearn, sys
print(f"python      {sys.version.split()[0]}")
print(f"numpy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")

Those version numbers are printed on purpose. Library versions were one of my
suspects for the original inconsistency, and a reader who gets different
results should be able to check whether their environment differs from mine.

## Load the data

`Path.cwd()` is printed because "file not found" is almost always a working-
directory problem rather than a filename problem.

In [ ]:
CSV = Path("insurance.csv")

if not CSV.exists():
    print(f"Not found. Working directory is: {Path.cwd()}")
    print(f"Files here: {sorted(p.name for p in Path.cwd().iterdir())}")
    raise FileNotFoundError(CSV)

df = pd.read_csv(CSV)
print(df.shape)
df.head()

In [ ]:
df.describe(include="all").T

## Build the model

The question my part of the project asked was whether smoking and a high BMI
*compound* rather than simply add up — so alongside the main effects there's an
interaction term, `smoker × BMI`. Categorical variables are one-hot encoded with
the first level dropped to avoid perfect collinearity between the dummy columns.

In [ ]:
def build_design_matrix(df):
    df = df.copy()
    df["smoker_bin"] = (df["smoker"] == "yes").astype(int)
    df["smoker_bmi"] = df["smoker_bin"] * df["bmi"]   # the interaction term

    X = pd.get_dummies(
        df[["age", "bmi", "children", "sex", "region", "smoker_bin", "smoker_bmi"]],
        columns=["sex", "region"],
        drop_first=True,
    ).astype(float)
    y = df["charges"].astype(float)
    return X, y


X, y = build_design_matrix(df)
print(f"{X.shape[0]} rows, {X.shape[1]} predictors")
list(X.columns)

## One split, twice

`random_state=None` is deliberate here — this is the bug, reproduced on purpose.
Run the next cell, then run it again.

In [ ]:
def fit_once(X, y, test_size=0.2, random_state=None):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return LinearRegression().fit(X_tr, y_tr).score(X_te, y_te)


print(f"first  run: {fit_once(X, y):.4f}")
print(f"second run: {fit_once(X, y):.4f}")

Same code, same data, same cell — two different numbers. Nothing has been
modified between them. This is the whole problem in two lines.

## Two hundred splits

One number tells you nothing about how stable it is. Two hundred tells you the
shape of the thing you were reporting.

In [ ]:
N_SPLITS = 200

scores = np.array([fit_once(X, y) for _ in range(N_SPLITS)])

lo, hi = np.percentile(scores, [2.5, 97.5])
print(f"n splits      : {len(scores)}")
print(f"min / max     : {scores.min():.4f} / {scores.max():.4f}")
print(f"spread        : {scores.max() - scores.min():.4f}")
print(f"mean (sd)     : {scores.mean():.4f} ({scores.std():.4f})")
print(f"95% of splits : {lo:.4f} to {hi:.4f}")

## Where do my two numbers fall?

200 splits is enough to see the shape, but not enough to say anything precise
about the tails. For that, run it 10,000 times — it takes a few seconds and lets
me put an actual probability on each of the numbers I saw that night.

In [ ]:
MY_VALUES = [0.86, 0.91]
N_BIG = 10_000

big = np.array([fit_once(X, y) for _ in range(N_BIG)])

for v in MY_VALUES:
    pct = (big < v).mean() * 100
    hits = int((big >= v).sum())
    print(f"{v:.2f} -> {pct:5.1f}th percentile | reached in {hits:>5} of {N_BIG:,} splits")

## The chart

The two dashed lines are the numbers I actually saw that night, annotated with
what the 10,000 runs above say about each of them.

In [ ]:
BLUE, ORANGE, INK, GREY = "#4C6EF5", "#E8590C", "#212529", "#868E96"

# label text computed from the data, not typed in by hand
labels = {
    v: f"{v:.2f}\n{(big < v).mean() * 100:.0f}th pct"
       if (big >= v).sum() > N_BIG * 0.05
       else f"{v:.2f}\n{int((big >= v).sum())} in {N_BIG:,}"
    for v in MY_VALUES
}

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(scores, bins=30, color=BLUE, alpha=0.8, edgecolor="white", linewidth=0.6)

# add headroom first, so annotations can never collide with the bars
ax.set_ylim(0, ax.get_ylim()[1] * 1.22)
top = ax.get_ylim()[1]

for v in MY_VALUES:
    ax.axvline(v, color=ORANGE, linestyle="--", linewidth=1.8, zorder=3)
    ax.annotate(
        labels[v], xy=(v, top * 0.97), ha="center", va="top",
        fontsize=10, color=ORANGE, fontweight="bold", linespacing=1.4,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=ORANGE, lw=0.8, alpha=0.95),
        zorder=4,
    )

ax.axvline(scores.mean(), color=INK, linewidth=1.4, zorder=3)
ax.annotate(
    f"mean {scores.mean():.3f}", xy=(scores.mean(), top * 0.62),
    xytext=(-8, 0), textcoords="offset points", ha="right", va="center",
    fontsize=9, color=INK,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.9), zorder=4,
)

ax.set_xlabel("Test-set R²", fontsize=10)
ax.set_ylabel(f"Number of splits (out of {N_SPLITS})", fontsize=10)
ax.set_title("The same model, the same data, 200 different random splits",
             fontsize=13, pad=16, loc="left")
ax.text(0, 1.015,
        "Medical cost dataset · linear regression with smoker × BMI interaction · unseeded splits",
        transform=ax.transAxes, fontsize=9, color=GREY, va="bottom")

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#CED4DA")
ax.tick_params(colors="#495057", labelsize=9)
ax.grid(axis="y", alpha=0.25, linewidth=0.7)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig("r2_distribution.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

The label logic switches automatically: a value inside the bulk of the
distribution gets a percentile, a value out in the tail gets a raw count, because
"99.8th percentile" is a much weaker way of saying "18 times in 10,000".

`facecolor="white"` on the save matters — without it the PNG has a transparent
background, which turns the axis labels invisible against a dark page.

## What I should have reported

The fix isn't to pick a better single split. It's to stop depending on one.
Ten-fold cross-validation fits the model ten times on different partitions and
averages the result, so no single lucky slice can flatter you — and with a fixed
seed it returns the same answer every time.

In [ ]:
cv = cross_val_score(LinearRegression(), X, y, cv=10, scoring="r2")

print(f"10-fold CV scores : {np.round(cv, 3)}")
print(f"mean (sd)         : {cv.mean():.4f} ({cv.std():.4f})")
print()
print(f"single-split range: {scores.min():.3f} to {scores.max():.3f}")
print(f"CV fold range     : {cv.min():.3f} to {cv.max():.3f}")

Note that the individual folds still vary. Cross-validation doesn't remove the
variation — it *reports* it, which is the honest version. The spread across folds
is itself information about how confident you're entitled to be.

## Takeaway

A number that moves when you re-run it isn't a finding. Pin every source of
randomness — the split *and* the model. Restart the kernel and run top to bottom
before trusting anything. Report an average across folds, with its spread, rather
than a single split you happened to like.

The same discipline is what separates a real backtest from a lucky one — which is
where I'm taking this next.